# Maritime Chokepoints Under Conflict: Visual Data Analysis
Data Collection, Aggregation and Preprocessing Course (HS26 Group 107)

Current notebook evaluates three core Research Questions:
1. **Hormuz Under Fire**: Traffic Response to Regional Escalation
2. **Around the Cape**: Conflict Thresholds for Rerouting Away from the Red Sea
3. **Who Stays, Who Flees?**: Vessel-Type Resilience to Conflict Escalation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# Plot styling configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['figure.titlesize'] = 14

DATA_DIR_PROCESSED = Path("data/processed")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Load the three curated datasets
ds1 = pd.read_parquet(DATA_DIR_PROCESSED / "dataset_1" / "dataset_1_hormuz.parquet")
ds2 = pd.read_parquet(DATA_DIR_PROCESSED / "dataset_2" / "dataset_2_rerouting.parquet")
ds3 = pd.read_parquet(DATA_DIR_PROCESSED / "dataset_3" / "dataset_3_vessel_resilience.parquet")

print("Datasets loaded successfully:")
print(f"  Dataset 1 (Hormuz): {ds1.shape}")
print(f"  Dataset 2 (Cape Rerouting): {ds2.shape}")
print(f"  Dataset 3 (Vessel Resilience): {ds3.shape}")

## Research Question 1: Hormuz Under Fire - Traffic Response to Regional Escalation
*How strongly and how quickly do weekly transit counts (n_total) and capacity (DWT) through the Strait of Hormuz decline after weeks of elevated conflict activity, and how many weeks does traffic take to return to its pre-escalation baseline?*

We use a **dual-axis time series plot** combining total weekly transits, metric deadweight tonnage (capacity), and a 4-week rolling conflict intensity index.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6), dpi=300)

# Primary Y-axis: PortWatch Shipping Transits & Deadweight Tonnage
color_transits = '#1f77b4'
color_capacity = '#2ca02c'

line1 = ax1.plot(ds1['week_start'], ds1['n_total'], color=color_transits, linewidth=2, label='Weekly Transits ($n_{total}$)')
ax1.set_xlabel('Timeline (Week Start)', fontweight='bold', labelpad=10)
ax1.set_ylabel('Total Ship Transits (Calls / Week)', color=color_transits, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_transits)
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Secondary Y-axis: Conflict Intensity
ax2 = ax1.twinx()
color_conflict = '#d62728'
line2 = ax2.plot(ds1['week_start'], ds1['conflict_events_rolling4w'], color=color_conflict, linewidth=2, linestyle='--', label='Rolling 4-Week Conflict Incidents (ACLED)')
ax2.fill_between(ds1['week_start'], ds1['conflict_events_rolling4w'], color=color_conflict, alpha=0.15)
ax2.set_ylabel('Weekly Conflict Events (4-Week Rolling Mean)', color=color_conflict, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color_conflict)
ax2.grid(False)  # Avoid overlapping gridlines

# Combine legends from both axes
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='lower left', frameon=True)

plt.title('Figure 1: Strait of Hormuz Vessel Transits vs. Regional Conflict Escalation (2023–2026)', pad=15)
fig.tight_layout()
plt.savefig(FIGURES_DIR / "fig1_hormuz_escalation.png")
plt.show()

### Supplementary Analysis for RQ1: Cross-Correlation / Lag Analysis
To determine the exact number of weeks required for traffic to respond to conflict peaks, we compute the Pearson correlation between conflict intensity at time *t* and vessel transits at lag *t+k* (from -8 to +8 weeks).

In [ ]:
lags = range(-8, 9)
corrs = [ds1['conflict_events_rolling4w'].corr(ds1['n_total'].shift(lag)) for lag in lags]

plt.figure(figsize=(9, 4), dpi=300)
colors = ['#d62728' if c == min(corrs) else '#1f77b4' for c in corrs]
bars = plt.bar(lags, corrs, color=colors, edgecolor='black', alpha=0.85)

min_lag = lags[np.argmin(corrs)]
min_val = min(corrs)
plt.axvline(0, color='gray', linestyle=':', linewidth=1)
plt.axhline(0, color='black', linewidth=0.8)

plt.title(f'Lagged Correlation: Rolling Conflict (t) vs. Hormuz Transits (t + k)\nMaximum Impact at Lag k = {min_lag} Weeks (r = {min_val:.2f})', pad=12)
plt.xlabel('Lag k (Weeks After Escalation)', fontweight='bold')
plt.ylabel('Pearson Correlation (r)', fontweight='bold')
plt.xticks(lags)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "fig1_sub_hormuz_lag_correlation.png")
plt.show()

## Research Question 3: Who Stays, Who Flees? — Vessel-Type Resilience
*How does weekly transit change differ between container ships, tankers, and dry bulk carriers under High vs. Low conflict conditions? Which vessel type retains the largest share of baseline traffic?*

Using Seaborn, we compare normalized vessel retention ratios:
$$\text{Retention Ratio} = \frac{n_{\text{vessel}, t}}{n_{\text{vessel}, \text{baseline}}}$$
across the Strait of Hormuz and Bab el-Mandeb Strait under different conflict regimes.

In [ ]:
# Melt the retention columns into long format for Seaborn bar plotting
vessel_retention = ds3.melt(
    id_vars=['portname', 'conflict_intensity_level'],
    value_vars=['retention_container', 'retention_tanker', 'retention_dry_bulk'],
    var_name='Vessel Type',
    value_name='Retention Ratio'
)

# Clean labels for presentation
vessel_retention['Vessel Type'] = vessel_retention['Vessel Type'].replace({
    'retention_container': 'Container Ships',
    'retention_tanker': 'Tankers',
    'retention_dry_bulk': 'Dry Bulk Carriers'
})

g = sns.catplot(
    data=vessel_retention,
    x='Vessel Type',
    y='Retention Ratio',
    hue='conflict_intensity_level',
    col='portname',
    kind='bar',
    palette={'Low': '#2b83ba', 'High': '#d7191c'},
    height=5,
    aspect=1.2,
    edgecolor='black',
    errorbar='se',
    capsize=0.08
)

g.set_axis_labels("", "Traffic Retention Relative to Baseline ($1.0 = 100\\%$)")
g.set_titles(col_template="{col_name}", weight='bold')

# Add 1.0 baseline reference line to all subplots
for ax in g.axes.flat:
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.set_ylim(0, 1.4)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

g._legend.set_title("Conflict Regime")
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle('Figure 3: Vessel-Type Retention by Maritime Strait Under High vs. Low Conflict Regimes', fontsize=13, weight='bold')

plt.savefig(FIGURES_DIR / "fig3_vessel_resilience.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Generate numerical summary table for report discussion
summary_table = ds3.groupby(['portname', 'conflict_intensity_level'])[['retention_container', 'retention_tanker', 'retention_dry_bulk']].mean() * 100
summary_table = summary_table.round(1).rename(columns={
    'retention_container': 'Container Retention (%)',
    'retention_tanker': 'Tanker Retention (%)',
    'retention_dry_bulk': 'Dry Bulk Retention (%)'
})

display(summary_table)
summary_table.to_csv("figures/table_vessel_retention_summary.csv")
print("All charts and summary table exported to /figures folder.")